# Docling for Dutch administrative-decision PDFs

This notebook converts PDFs with **Docling**, saves the full structured Docling output, lists detected headings, and creates a simple baseline for locating operative sections such as **Besluit**, **Beslissing**, and **Dictum**.

### How to use this notebook
1. Run the cells **from top to bottom**.
2. The first code cell installs the packages for you.
3. Upload or place your PDFs when the notebook asks you to.
4. Run the batch-conversion cell.
5. Inspect the headings and candidate operative sections.

The notebook deliberately keeps **PDF parsing** and **legal section selection** separate. The Docling JSON should be kept as the canonical parsed representation for later experiments.

In [1]:
import os
import subprocess

vs_path = r"C:\\Program Files (x86)\\Microsoft Visual Studio\\18\\BuildTools\\VC\\Auxiliary\\Build\\vcvars64.bat"

result = subprocess.run(
    f'"{vs_path}" && set',
    shell=True,
    capture_output=True,
    text=True
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

In [2]:
import sys
print(sys.executable)

c:\Users\Gebruiker\AppData\Local\Python\pythoncore-3.14-64\python.exe


## 1. Install the packages

Run the cell below once. In Google Colab this is usually all you need. In a local Jupyter notebook, it installs Docling into the Python environment used by the notebook.

If Jupyter asks you to restart the kernel after installation, restart it and continue with the next cell.

In [3]:
#%pip install -q docling pandas

## 2. Imports and folders

The notebook creates:

- `pdfs/` — input PDFs
- `docling_output/` — Markdown, JSON, heading lists, and operative-section candidates

In [8]:
from pathlib import Path
import json
import re
import shutil
import traceback
import zipfile

import pandas as pd

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption


INPUT_PATH = Path(r"E:\\CITaDOG\\PDFs\\Rijksoverheid")
OUTPUT_DIR = Path(r"E:\\CITaDOG\\docling_output\\Rijksoverheid")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# If path doesn't exist, check whether it is actually a .zip
# ---------------------------------------------------------

if not INPUT_PATH.exists():
    zip_candidate = INPUT_PATH.with_suffix(".zip")

    if zip_candidate.exists():
        INPUT_PATH = zip_candidate
        print(f"Found ZIP file: {INPUT_PATH}")
    else:
        raise FileNotFoundError(
            f"Could not find either:\n"
            f"  {INPUT_PATH}\n"
            f"or:\n"
            f"  {zip_candidate}"
        )


# ---------------------------------------------------------
# Handle normal folder OR ZIP file
# ---------------------------------------------------------

if INPUT_PATH.is_dir():

    INPUT_DIR = INPUT_PATH
    print("Input type   : Folder")


elif INPUT_PATH.is_file() and zipfile.is_zipfile(INPUT_PATH):

    EXTRACT_DIR = INPUT_PATH.parent / f"{INPUT_PATH.stem}_extracted"

    # Remove previous extraction
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    print("Input type   : ZIP file")
    print("ZIP file     :", INPUT_PATH.resolve())
    print("Extracting to:", EXTRACT_DIR.resolve())

    with zipfile.ZipFile(INPUT_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

    INPUT_DIR = EXTRACT_DIR


else:
    raise ValueError(
        f"Input exists, but is neither a folder nor a valid ZIP:\n{INPUT_PATH}"
    )


print("\nInput folder :", INPUT_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())


# ---------------------------------------------------------
# Find PDFs
# ---------------------------------------------------------

pdf_files = sorted(INPUT_DIR.rglob("*.pdf"))

print(f"\nFound {len(pdf_files)} PDF(s):")

for p in pdf_files[:20]:
    print(" -", p)

if len(pdf_files) > 20:
    print(f" ... and {len(pdf_files) - 20} more")

Input type   : Folder

Input folder : E:\CITaDOG\PDFs\Rijksoverheid
Output folder: E:\CITaDOG\docling_output\Rijksoverheid

Found 135 PDF(s):
 - E:\CITaDOG\PDFs\Rijksoverheid\0_Wijzigingsbeschikking-subsidie-Nationaal-Onderwijsmuseum-2025.pdf
 - E:\CITaDOG\PDFs\Rijksoverheid\100_Beschikking-aanvraag-subsidie-voor-het-project-Specialistische-Kennisborging.pdf
 - E:\CITaDOG\PDFs\Rijksoverheid\101_Subsidieverlening-over-deelproject-STD2-Thermische-en-Pneumatische-Systemen-(TePS).pdf
 - E:\CITaDOG\PDFs\Rijksoverheid\102_Subsidieverlening-over-deelproject-STD1-Elektrische-Kabelsystemen.pdf
 - E:\CITaDOG\PDFs\Rijksoverheid\103_Besluit-op-subsidieverlening-Huurders-in-energiearmoede.pdf
 - E:\CITaDOG\PDFs\Rijksoverheid\104_Beschikking-verlening-subsidie-voor-Rotterdams-Philharmonisch-Orkest-X-The-Ukraine-Liberation-Orchestra.pdf
 - E:\CITaDOG\PDFs\Rijksoverheid\105_Verlening-Steunaanvraag-UUBF-2023.pdf
 - E:\CITaDOG\PDFs\Rijksoverheid\106_Definitief-besluit-incidentele-subsidie-CCS-project-Ya

## 3. Add your PDFs

### If you use Google Colab
Run the next cell and select one or more PDFs from your computer.

### If you use Jupyter locally
Put your PDFs in the `pdfs` folder next to this notebook. You may also create subfolders such as `pdfs/ACM/`, `pdfs/AP/`, or `pdfs/AFM/`.

The cell below detects Colab automatically. Outside Colab it simply shows which PDFs it can see.

## 4. Configure Docling

`do_ocr = True` leaves OCR available for image/scanned material while Docling still handles normal text PDFs. For a final experiment, record the Docling version and keep the same configuration for every input condition.

In [9]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True

converter = DocumentConverter(
    allowed_formats=[InputFormat.PDF],
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    },
)

print("Docling converter is ready.")

Docling converter is ready.


## 5. Define the legal-heading baseline

This is intentionally a **simple baseline**. It first asks Docling which blocks are section headers and then checks whether the normalized heading is one of your operative headings.

You should inspect the detected headings on a development sample before expanding this vocabulary.

In [10]:
OPERATIVE_HEADINGS = {
    "besluit",
    "beslissing",
    "dictum",
    "beslispunten",
    "slotsom",
    "slotsom en besluit",
    "conclusie en besluit",
}


def normalize_heading(text: str) -> str:
    """Normalize common numbered Dutch headings."""
    text = (text or "").strip().lower()

    # Numeric prefixes: 5., 5.1, 5.1.2 etc.
    text = re.sub(r"^\s*\d+(?:\.\d+)*\.?\s*", "", text)

    # Roman-numeral prefixes: IV. / IV)
    text = re.sub(r"^\s*[ivxlcdm]+[\.\)]\s*", "", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+", " ", text)
    return text.strip(" :-–—.")


def label_name(item) -> str:
    """Return a stable string representation of a Docling item label."""
    label = getattr(item, "label", "")
    return str(getattr(label, "value", label)).lower()


def is_section_header(item) -> bool:
    return label_name(item) == "section_header"


def get_page_number(item):
    """Return the first provenance page number, when available."""
    prov = getattr(item, "prov", None)
    if prov:
        return getattr(prov[0], "page_no", None)
    return None


def get_heading_level(item):
    """Docling section headers may contain a hierarchy level."""
    level = getattr(item, "level", None)
    return level if isinstance(level, int) else 1


def extract_headings(doc):
    headings = []

    for item, tree_level in doc.iterate_items():
        if not is_section_header(item):
            continue

        text = getattr(item, "text", "") or ""
        headings.append({
            "text": text,
            "normalized": normalize_heading(text),
            "heading_level": get_heading_level(item),
            "tree_level": tree_level,
            "page": get_page_number(item),
            "self_ref": getattr(item, "self_ref", None),
        })

    return headings


def extract_operative_sections(doc):
    """
    Baseline: find an explicit operative heading and collect following
    textual items until the next section header at the same or a higher level.
    """
    items = list(doc.iterate_items())
    results = []

    for i, (item, tree_level) in enumerate(items):
        if not is_section_header(item):
            continue

        heading = (getattr(item, "text", "") or "").strip()
        normalized = normalize_heading(heading)

        if normalized not in OPERATIVE_HEADINGS:
            continue

        start_level = get_heading_level(item)
        pages = set()
        content = []

        page = get_page_number(item)
        if page is not None:
            pages.add(page)

        for next_item, next_tree_level in items[i + 1:]:
            if is_section_header(next_item):
                next_level = get_heading_level(next_item)
                if next_level <= start_level:
                    break

            text = getattr(next_item, "text", None)
            if text and str(text).strip():
                content.append(str(text).strip())
                page = get_page_number(next_item)
                if page is not None:
                    pages.add(page)

        results.append({
            "heading": heading,
            "normalized_heading": normalized,
            "heading_level": start_level,
            "start_page": get_page_number(item),
            "pages": sorted(pages),
            "text": "\n\n".join(content),
        })

    return results

print("Functions loaded.")

Functions loaded.


## 6. Convert all PDFs

For each PDF this saves:

- `.md` — convenient human-readable conversion
- `.json` — full lossless Docling representation; **keep this for your research**
- `.headings.json` — headings detected by Docling
- `.operative.json` — candidate *Besluit/Dictum* section(s)

It also creates corpus-level CSV files for inspection.

In [ ]:
import shutil
import re
import unicodedata

TEMP_DIR = Path(r"C:\docling_tmp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)


def make_safe_temp_copy(pdf_path: Path) -> Path:
    name = unicodedata.normalize("NFKD", pdf_path.name)

    name = (
        name
        .replace("“", "")
        .replace("”", "")
        .replace("‘", "")
        .replace("’", "")
        .replace("–", "-")
        .replace("—", "-")
    )

    # Remove remaining non-ASCII characters
    name = name.encode("ascii", errors="ignore").decode("ascii")

    # Remove problematic Windows characters
    name = re.sub(r'[<>:"/\\|?*]', "_", name)

    temp_path = TEMP_DIR / name

    shutil.copy2(pdf_path, temp_path)

    return temp_path

def process_pdf(pdf_path: Path, skip_existing: bool = True):
    relative_path = pdf_path.relative_to(INPUT_DIR)
    output_subdir = OUTPUT_DIR / relative_path.parent
    output_subdir.mkdir(parents=True, exist_ok=True)

    stem = pdf_path.stem

    # Expected output files for this PDF
    md_path = output_subdir / f"{stem}.md"
    json_path = output_subdir / f"{stem}.json"
    headings_path = output_subdir / f"{stem}.headings.json"
    operative_path = output_subdir / f"{stem}.operative.json"

    # Only treat the PDF as already processed when all expected outputs exist.
    expected_outputs = [
        md_path,
        json_path,
        headings_path,
        operative_path,
    ]

    already_processed = all(path.exists() for path in expected_outputs)

    if skip_existing and already_processed:
        print(f"Skipping already converted: {pdf_path}")

        # Re-load the derived JSON files so the corpus-level CSVs
        # are still rebuilt correctly on every notebook run.
        with headings_path.open("r", encoding="utf-8") as f:
            headings = json.load(f)

        with operative_path.open("r", encoding="utf-8") as f:
            operative_sections = json.load(f)

        return headings, operative_sections, False

    print(f"Processing: {pdf_path}")

    temp_pdf = make_safe_temp_copy(pdf_path)

    try:
        result = converter.convert(temp_pdf)
        doc = result.document
    finally:
        if temp_pdf.exists():
            temp_pdf.unlink()

    # Full exports
    doc.save_as_markdown(md_path)
    doc.save_as_json(json_path)

    # Derived inspection data
    headings = extract_headings(doc)
    operative_sections = extract_operative_sections(doc)

    with headings_path.open("w", encoding="utf-8") as f:
        json.dump(headings, f, ensure_ascii=False, indent=2)

    with operative_path.open("w", encoding="utf-8") as f:
        json.dump(operative_sections, f, ensure_ascii=False, indent=2)

    return headings, operative_sections, True

all_headings = []
all_operatives = []
failures = []

processed_count = 0
skipped_count = 0

for pdf_path in pdf_files:
    try:
        headings, operatives, was_processed = process_pdf(pdf_path)

        if was_processed:
            processed_count += 1
        else:
            skipped_count += 1

        document_id = str(pdf_path.relative_to(INPUT_DIR))

        for row in headings:
            all_headings.append({"document": document_id, **row})

        for idx, row in enumerate(operatives, start=1):
            all_operatives.append({
                "document": document_id,
                "candidate_no": idx,
                **row,
            })

        print(
            f"  headings={len(headings)}, "
            f"operative candidates={len(operatives)}"
        )

    except Exception as exc:
        print(f"  ERROR: {exc}")
        failures.append({
            "file": str(pdf_path),
            "error": str(exc),
            "traceback": traceback.format_exc(),
        })

# Save corpus-level inspection files
headings_df = pd.DataFrame(all_headings)
operatives_df = pd.DataFrame(all_operatives)

headings_df.to_csv(OUTPUT_DIR / "all_headings.csv", index=False, encoding="utf-8-sig")
operatives_df.to_csv(OUTPUT_DIR / "operative_candidates.csv", index=False, encoding="utf-8-sig")

with (OUTPUT_DIR / "failures.json").open("w", encoding="utf-8") as f:
    json.dump(failures, f, ensure_ascii=False, indent=2)

print("\nFinished")
print("Newly converted:", processed_count)
print("Skipped existing:", skipped_count)
print("Failed:          ", len(failures))
print("Headings:        ", len(headings_df))
print("Candidates:      ", len(operatives_df))


pdf_files = sorted(
    p for p in INPUT_DIR.rglob("*.pdf")
    if p.name.upper().startswith("ANVS")
)

if not pdf_files:
    raise FileNotFoundError(
        "No PDFs found. Add PDFs in the previous step and run this cell again."
    )

all_headings = []
all_operatives = []
failures = []

for pdf_path in pdf_files:
    try:
        headings, operatives = process_pdf(pdf_path)
        document_id = str(pdf_path.relative_to(INPUT_DIR))

        for row in headings:
            all_headings.append({"document": document_id, **row})

        for idx, row in enumerate(operatives, start=1):
            all_operatives.append({
                "document": document_id,
                "candidate_no": idx,
                **row,
            })

        print(f"  headings={len(headings)}, operative candidates={len(operatives)}")

    except Exception as exc:
        print(f"  ERROR: {exc}")
        failures.append({
            "file": str(pdf_path),
            "error": str(exc),
            "traceback": traceback.format_exc(),
        })

# Save corpus-level inspection files
headings_df = pd.DataFrame(all_headings)
operatives_df = pd.DataFrame(all_operatives)

headings_df.to_csv(OUTPUT_DIR / "all_headings.csv", index=False, encoding="utf-8-sig")
operatives_df.to_csv(OUTPUT_DIR / "operative_candidates.csv", index=False, encoding="utf-8-sig")

with (OUTPUT_DIR / "failures.json").open("w", encoding="utf-8") as f:
    json.dump(failures, f, ensure_ascii=False, indent=2)

print("\nFinished")
print("Successful:", len(pdf_files) - len(failures))
print("Failed:    ", len(failures))
print("Headings:  ", len(headings_df))
print("Candidates:", len(operatives_df))

Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\0_Wijzigingsbeschikking-subsidie-Nationaal-Onderwijsmuseum-2025.pdf
  headings=9, operative candidates=1
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\100_Beschikking-aanvraag-subsidie-voor-het-project-Specialistische-Kennisborging.pdf
  headings=17, operative candidates=1
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\101_Subsidieverlening-over-deelproject-STD2-Thermische-en-Pneumatische-Systemen-(TePS).pdf
  headings=80, operative candidates=1
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\102_Subsidieverlening-over-deelproject-STD1-Elektrische-Kabelsystemen.pdf
  headings=111, operative candidates=1
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\103_Besluit-op-subsidieverlening-Huurders-in-energiearmoede.pdf
  headings=10, operative candidates=0
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\104_Beschikking-verlening-subsidie-voor-Rotterdams-Philharmonisch-Orkest-X-The-U

Input document E:\CITaDOG\PDFs\Rijksoverheid\114_Besluit-op-subsidieverzoek-“Veenweidensloot-van-de-toekomst-–-VIPNL-project”.pdf is not valid.


  headings=19, operative candidates=1
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\113_Besluit-op-verlening-subsidie-voor-project-Macaw.pdf
  headings=32, operative candidates=1
Processing: E:\CITaDOG\PDFs\Rijksoverheid\114_Besluit-op-subsidieverzoek-“Veenweidensloot-van-de-toekomst-–-VIPNL-project”.pdf
  ERROR: Conversion failed for: E:\CITaDOG\PDFs\Rijksoverheid\114_Besluit-op-subsidieverzoek-“Veenweidensloot-van-de-toekomst-–-VIPNL-project”.pdf with status: failure. Errors: docling-parse could not load document 2cce3b239d7a3054f01c7417dd2f1a56a0b0a62bf4d00bd3d972cdb6edf2c5a7: Failed to load document with key key=E:\CITaDOG\PDFs\Rijksoverheid\114_Besluit-op-subsidieverzoek-“Veenweidensloot-van-de-toekomst-–-VIPNL-project”.pdf
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\115_Subsidieverlening-Hydrogen-Aircraft-Powertrain-and-Storage-Systems.pdf
  headings=62, operative candidates=1
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\116_Beschikking-aanv

Input document E:\CITaDOG\PDFs\Rijksoverheid\122_Besluit-subsidieverzoek-project-‘NEWLIFE’.pdf is not valid.
Input document E:\CITaDOG\PDFs\Rijksoverheid\126_Besluit-subsidieverzoek-project-‘Hiconnects’.pdf is not valid.
Input document E:\CITaDOG\PDFs\Rijksoverheid\128_Besluit-subsidieverzoek-project-‘AGRARSENSE’.pdf is not valid.
Input document E:\CITaDOG\PDFs\Rijksoverheid\129_Besluit-subsidieverzoek-project-‘14ACMOS’.pdf is not valid.
Input document E:\CITaDOG\PDFs\Rijksoverheid\131_Besluit-subsidieverzoek-project-‘PowerizeD’.pdf is not valid.


  ERROR: Conversion failed for: E:\CITaDOG\PDFs\Rijksoverheid\122_Besluit-subsidieverzoek-project-‘NEWLIFE’.pdf with status: failure. Errors: docling-parse could not load document 91e5d84e898ee6dcafb0487153e1f3fbc5883d2f7de8a38b93bbc6408316396c: Failed to load document with key key=E:\CITaDOG\PDFs\Rijksoverheid\122_Besluit-subsidieverzoek-project-‘NEWLIFE’.pdf
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\123_Besluit-verlening-subsidie-voor-project-Rebecca.pdf
  headings=7, operative candidates=0
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\124_Besluit-verlening-subsidie-voor-project-A-IQ-Ready.pdf
  headings=6, operative candidates=0
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\125_Besluit-subsidieverzoek-project-CLEVER.pdf
  headings=6, operative candidates=0
Processing: E:\CITaDOG\PDFs\Rijksoverheid\126_Besluit-subsidieverzoek-project-‘Hiconnects’.pdf
  ERROR: Conversion failed for: E:\CITaDOG\PDFs\Rijksoverheid\126_Besluit-subsidieverzoek-proje

Input document E:\CITaDOG\PDFs\Rijksoverheid\86_Besluit-wijziging-projectsubsidie-‘Versnellen-en-verbreden-gebruik-netwerkvoorzieningen-Netwerk-Digitaal-Erfgoed’.pdf is not valid.
Input document E:\CITaDOG\PDFs\Rijksoverheid\94_Besluit-op-subsidieverzoek-“Kennisverspreiding-Agroforestry-–-Kennis-op-Maat-project.pdf is not valid.


  headings=19, operative candidates=1
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\82_Besluit-op-subsidieverzoek-project-Tropische-potplanten-koeler-winter-door-onder-LED-verlichting.pdf
  headings=15, operative candidates=1
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\83_Rectificatie-subsidieverlening-Energiebesparing-ronde-2.pdf
  headings=9, operative candidates=0
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\84_Rectificatie-subsidieverlening-Werklandschappen-van-de-Toekomst-2023-2024.pdf
  headings=8, operative candidates=0
Skipping already converted: E:\CITaDOG\PDFs\Rijksoverheid\85_Besluit-verlening-projectsubsidie-Versterken-onderzoekscapaciteit-in-2024.pdf
  headings=19, operative candidates=1
Processing: E:\CITaDOG\PDFs\Rijksoverheid\86_Besluit-wijziging-projectsubsidie-‘Versnellen-en-verbreden-gebruik-netwerkvoorzieningen-Netwerk-Digitaal-Erfgoed’.pdf
  ERROR: Conversion failed for: E:\CITaDOG\PDFs\Rijksoverheid\86_Besluit-wijziging-proje

Input document E:\CITaDOG\PDFs\Rijksoverheid\95_Besluit-op-subsidieverzoek-“Versterken-biodiversiteit-op-melkveebedrijven-door-de-kennisverspreiding-van-de-KPI’s-Natuur-en-landschap-en-Kruidenrijk-grasland”.pdf is not valid.
Input document E:\CITaDOG\PDFs\Rijksoverheid\96_Besluit-op-subsidieverzoek-“Innovatieplatform-Akkerbouw-2024-2027”.pdf is not valid.


Processing: E:\CITaDOG\PDFs\Rijksoverheid\95_Besluit-op-subsidieverzoek-“Versterken-biodiversiteit-op-melkveebedrijven-door-de-kennisverspreiding-van-de-KPI’s-Natuur-en-landschap-en-Kruidenrijk-grasland”.pdf
  ERROR: Conversion failed for: E:\CITaDOG\PDFs\Rijksoverheid\95_Besluit-op-subsidieverzoek-“Versterken-biodiversiteit-op-melkveebedrijven-door-de-kennisverspreiding-van-de-KPI’s-Natuur-en-landschap-en-Kruidenrijk-grasland”.pdf with status: failure. Errors: docling-parse could not load document 07ad8aa6d753d62366ce94858e99b7209ac4ce180d0b468b77f90d220a7718d3: Failed to load document with key key=E:\CITaDOG\PDFs\Rijksoverheid\95_Besluit-op-subsidieverzoek-“Versterken-biodiversiteit-op-melkveebedrijven-door-de-kennisverspreiding-van-de-KPI’s-Natuur-en-landschap-en-Kruidenrijk-grasland”.pdf
Processing: E:\CITaDOG\PDFs\Rijksoverheid\96_Besluit-op-subsidieverzoek-“Innovatieplatform-Akkerbouw-2024-2027”.pdf
  ERROR: Conversion failed for: E:\CITaDOG\PDFs\Rijksoverheid\96_Besluit-op-subsi

FileNotFoundError: No PDFs found. Add PDFs in the previous step and run this cell again.

## 7. Inspect Docling's detected headings

This is the first thing I recommend checking on a sample of your corpus. It tells you whether errors originate in PDF/layout parsing or in your own legal heading vocabulary.

In [17]:
if headings_df.empty:
    print("No section headers were detected.")
else:
    display_cols = [
        c for c in [
            "document", "page", "text", "normalized",
            "heading_level", "tree_level"
        ] if c in headings_df.columns
    ]
    display(headings_df[display_cols].head(100))

,document,page,text,normalized,heading_level,tree_level
0,1071e_charlois_Vergunning.pdf,2,Bijlage(n) behorende bij de evenementenvergunning,bijlage(n) behorende bij de evenementenvergunning,1,1
1,1071e_charlois_Vergunning.pdf,2,Niet eens met deze beslissing?,niet eens met deze beslissing?,1,1
2,1071e_charlois_Vergunning.pdf,2,De Burgemeester,de burgemeester,1,1
3,1071e_charlois_Vergunning.pdf,3,Bijlage I; Voorschriften behorende bij de even...,bijlage i; voorschriften behorende bij de even...,1,1
4,1071e_charlois_Vergunning.pdf,3,Aanvullende voorwaarden,aanvullende voorwaarden,1,1
...,...,...,...,...,...,...
95,1255e_delfshaven_Vergunning.pdf,5,Afval en milieu,afval en milieu,1,1
96,1255e_delfshaven_Vergunning.pdf,5,Aansprakelijkheid,aansprakelijkheid,1,1
97,1255e_delfshaven_Vergunning.pdf,6,Bijlage II: geluidsvoorschriften bij de evenem...,bijlage ii: geluidsvoorschriften bij de evenem...,1,1
98,1255e_delfshaven_Vergunning.pdf,6,Algemene zorgplicht,algemene zorgplicht,1,1


## 8. Inspect candidate operative sections

A candidate is created only when Docling detects a section header whose normalized text exactly matches the current operative-heading vocabulary.

In [ ]:
if operatives_df.empty:
    print("No operative-section candidates found.")
    print("Inspect all_headings.csv before adding more heading variants.")
else:
    preview = operatives_df.copy()
    preview["text_preview"] = preview["text"].fillna("").str.slice(0, 700)
    display_cols = [
        c for c in [
            "document", "candidate_no", "start_page", "heading",
            "normalized_heading", "text_preview"
        ] if c in preview.columns
    ]
    display(preview[display_cols].head(100))

## 9. Optional: see which documents have no candidate *Besluit/Dictum* section

These documents are especially useful for your error analysis. A missing candidate can mean:

1. Docling failed to identify the heading;
2. the document uses a heading that is not yet in your vocabulary; or
3. there is no explicit operative heading and you need a semantic/lexical fallback.

In [ ]:
documents = pd.DataFrame({
    "document": [str(p.relative_to(INPUT_DIR)) for p in pdf_files]
})

if operatives_df.empty:
    documents["has_operative_candidate"] = False
else:
    candidate_docs = set(operatives_df["document"])
    documents["has_operative_candidate"] = documents["document"].isin(candidate_docs)

print(documents["has_operative_candidate"].value_counts(dropna=False))
display(documents[~documents["has_operative_candidate"]].head(100))

## 10. Create a ZIP of all outputs

Run this after processing. It packages `docling_output/` so you can keep or download the results.

In [ ]:
#zip_path = Path("docling_output.zip")
#if zip_path.exists():
#    zip_path.unlink()

#shutil.make_archive("docling_output", "zip", OUTPUT_DIR)
#print("Created:", zip_path.resolve())

#try:
#    from google.colab import files
#    files.download(str(zip_path))
#except ImportError:
#    print("Not running in Colab. The ZIP is stored next to the notebook.")

## Recommended research workflow after this notebook

Do **not** immediately add many rules when a document fails. First manually inspect approximately 50–100 heterogeneous PDFs and classify each failure as:

- PDF/OCR/layout failure;
- section-header recognition failure;
- unseen heading wording; or
- no explicit heading.

Then freeze a development vocabulary and evaluate it on a separate annotated test set. Keep the `.json` files so that later full-text, section-based, and sentence-based conditions all originate from the same Docling parse.